# relu-elementwise-max — ex1: implement ReLU + verify the derivative jump at 0

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `relu-elementwise-max`. Running the final beacon cell reports progress against the `CNN: ReLU as elementwise max` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: ReLU as elementwise max` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`relu-elementwise-max`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "relu-elementwise-max"
DD_SUBTOPIC = "CNN: ReLU as elementwise max"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## ReLU as elementwise max — quick refresher

`ReLU(x) = max(x, 0)` — applied **elementwise** to every entry of the input tensor. In PyTorch:

```
y = t.maximum(x, t.tensor(0.0))      # the canonical form
y = x.clamp(min=0)                   # equivalent
y = F.relu(x)                        # library form
```

**Read the math:** Negative entries become 0; non-negative entries pass through unchanged. The function is piecewise-linear with a kink at `x = 0`.

**Derivative.** `dReLU/dx = 1 if x > 0 else 0`. At `x = 0` the derivative is undefined — there's a **sub-gradient** anywhere in `[0, 1]` and the convention depends on which PyTorch op you use:

- `t.maximum(x, t.tensor(0.0))` → grad **0.5** at `x = 0` (the symmetric average — `maximum` distributes equally on ties).
- `F.relu(x)` → grad **0** at `x = 0` (the smallest sub-gradient, PyTorch's `relu` convention).

Both are valid sub-gradients; both work for training. The asymmetry matters only at the measure-zero point `x = 0`.

**Why the simple max.** ReLU is cheap (one comparison, one branch), it doesn't saturate for large positive inputs (unlike sigmoid/tanh), and its gradient is exactly 1 in the active region — three properties that make it the default activation for hidden layers since 2010.

### Exercise 1 — implement ReLU + verify the derivative jump at 0

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply `ReLU(x) = max(x, 0)` elementwise via `t.maximum`, then use autograd to verify the derivative is 1 for x > 0, 0 for x < 0, and 0.5 at x = 0 (the symmetric sub-gradient of `t.maximum` on ties).
> Keywords: relu, elementwise, derivative, autograd
> ```

**KCs targeted:** `relu-max-with-zero`, `relu-derivative-at-zero`

Implement `ex1_relu_and_grad(x)`. Given a 1-D tensor `x: (N,)` with `requires_grad=True` already set, compute `y = ReLU(x)` (using `t.maximum`, NOT `F.relu`), then use `t.autograd.grad` to compute `dy_sum/dx` where `y_sum = y.sum()`. Return `(y, dy_dx)` — each shape `(N,)`.

**Required implementation.** Use `t.maximum(x, t.tensor(0.0))` for the forward pass — this drill targets the canonical elementwise-max form. Equivalent options (`x.clamp(min=0)`, `F.relu(x)`) are forbidden so the drill exercises the *definition*, not the library shortcut.

**For the gradient:**
```
(grad,) = t.autograd.grad(y.sum(), x)
```
This computes `d(sum(y))/dx = dy/dx` (since summing then differentiating w.r.t. each component yields a vector `(dy_0/dx_0, dy_1/dx_1, ...)` — exactly the per-element derivative).

**The verification.** The test confirms:
- For positive inputs, grad is 1.
- For negative inputs, grad is 0.
- At exactly x=0, grad is **0.5** (the symmetric sub-gradient that `t.maximum` produces on ties — different from `F.relu`, which uses 0). Both are valid sub-gradients in the closed interval `[0, 1]`.

In [ ]:
def ex1_relu_and_grad(x: Tensor):
    y = t.maximum(x, t.tensor(0.0))
    (dy_dx,) = t.autograd.grad(y.sum(), x)
    return y, dy_dx


<details><summary>Solution</summary>

```python
def ex1_relu_and_grad(x: Tensor):
    y = t.maximum(x, t.tensor(0.0))
    (dy_dx,) = t.autograd.grad(y.sum(), x)
    return y, dy_dx
```

**Why `t.maximum` and not `t.max`.** `t.max(x, t.tensor(0.0))` is the same elementwise op, but `t.max(x)` (single arg) is the *reduction* — it returns a scalar (the global max), which is not what we want. `t.maximum(a, b)` is unambiguously elementwise.

**The derivative jump at 0.** Mathematically the derivative is undefined at `x = 0` (left-derivative is 0, right-derivative is 1). The op you use decides the sub-gradient convention:

- `t.maximum(x, 0)` → grad `0.5` at ties (symmetric average — this is what THIS drill exercises).
- `F.relu(x)` → grad `0` at ties (smallest sub-gradient).

Both choices live inside the valid sub-gradient interval `[0, 1]`. In practice the discrepancy is harmless because exact zeros are measure-zero events for random init.

**Why we test with NaN.** A common 'optimization' is to write ReLU as `x * (x > 0)`, which produces 0 at NaN inputs (since `nan > 0` is `False`). The canonical `t.maximum` form propagates NaN, which is the correct IEEE-754 behavior and what `F.relu` does too. So the NaN test is implicitly an 'are you using the right form' check.

**Why bother teaching this.** Most CNN bugs come from the layers between ReLUs (BN, conv layout). But a stray ReLU on the wrong tensor — final-classifier output, attention logits, gates — silently kills the negatives and breaks training. The `no-relu-on-final-layer` drill goes deeper into that failure.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()